<a href="https://www.kaggle.com/code/mrrogueknight/vandermonde-polynomial-solver-tarpeen-data?scriptVersionId=335768174" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [14]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [15]:
"""
Shifted Vandermonde Polynomial Interpolation and Extrapolation Framework

This module implements polynomial interpolation using shifted and scaled 
Vandermonde matrices, with adaptive extrapolation capabilities, numerical 
diagnostics, and comprehensive visualization.

Features:
    - Shifted and scaled Vandermonde basis for numerical stability
    - Exact interpolation and least-squares fitting
    - Adaptive extrapolation with polynomial capping
    - Comprehensive numerical diagnostics
    - Reliability scoring for predictions
    - Confidence intervals for extrapolation
    - Interactive data point manipulation
    - Real-time polynomial plotting with Matplotlib
    - Extrapolation visualization with uncertainty bands

Author: Scientific Computing Framework
License: MIT
"""

from __future__ import annotations

import numpy as np
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional, Tuple, List, Union, Any, Dict
from collections import deque
import logging
from math import comb
import time
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import LineCollection
import matplotlib as mpl

# Set matplotlib style for professional plots
plt.style.use('seaborn-v0_8-darkgrid')
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif']
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.labelsize'] = 11
mpl.rcParams['axes.titlesize'] = 12
mpl.rcParams['legend.fontsize'] = 9
mpl.rcParams['figure.titlesize'] = 13

# ===================================================================
# LOGGING
# ===================================================================

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

# ===================================================================
# CONSTANTS
# ===================================================================

MACHINE_EPSILON = np.finfo(np.float64).eps
DEFAULT_DEGREE = 6
MAX_HISTORY = 10
PLOT_POINTS = 500
PLOT_MARGIN = 0.3

CONDITION_SAFE = 1e6
CONDITION_WARNING = 1e10
CONDITION_CRITICAL = 1e14

# ===================================================================
# TYPE ALIASES
# ===================================================================

FloatArray = np.ndarray
Coefficients = np.ndarray

# ===================================================================
# ENUMS
# ===================================================================

class SolverStatus(Enum):
    """Solver status enumeration."""
    IDLE = "idle"
    SUCCESS = "success"
    WARNING = "warning"
    ERROR = "error"

class ShiftMethod(Enum):
    """Shift method enumeration."""
    MEAN = "mean"
    FIRST = "first"
    NONE = "none"

class ExpansionMode(Enum):
    """Expansion mode for polynomial display."""
    SHIFTED = "shifted"
    EXPANDED = "expanded"
    BOTH = "both"

class ExtrapolationMethod(Enum):
    """Extrapolation method enumeration."""
    LINEAR = "linear"
    POLYNOMIAL = "polynomial"
    RATIONAL = "rational"
    ENSEMBLE = "ensemble"
    BLENDED = "blended"
    ADAPTIVE = "adaptive"

class ReliabilityLevel(Enum):
    """Reliability levels for predictions."""
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"
    VERY_LOW = "very_low"
    UNRELIABLE = "unreliable"

# ===================================================================
# DATA CLASSES
# ===================================================================

@dataclass(slots=True)
class SingularValues:
    """Singular value diagnostics."""
    largest: float
    smallest: float
    ratio: float
    log10_ratio: float
    effective_rank: int
    
    @classmethod
    def from_svd(cls, singular_values: FloatArray, tolerance: float = MACHINE_EPSILON) -> SingularValues:
        """Create singular value diagnostics from SVD output."""
        sv = singular_values.copy()
        largest = sv[0]
        
        tol = max(sv) * tolerance * 10
        effective_rank = np.sum(sv > tol)
        
        if effective_rank > 0 and effective_rank <= len(sv):
            smallest = sv[effective_rank - 1]
            if len(sv) > effective_rank and sv[effective_rank] > 0:
                smallest = min(smallest, sv[effective_rank])
        else:
            smallest = sv[-1] if len(sv) > 0 else 0.0
        
        ratio = largest / (smallest + MACHINE_EPSILON)
        
        return cls(
            largest=float(largest),
            smallest=float(smallest),
            ratio=float(ratio),
            log10_ratio=np.log10(max(ratio, 1.0)),
            effective_rank=int(effective_rank)
        )

@dataclass(slots=True)
class NumericalDiagnostics:
    """Comprehensive numerical diagnostics."""
    condition_number: float
    condition_number_log10: float
    digits_lost: float
    residual_norm_l1: float
    residual_norm_l2: float
    residual_norm_linf: float
    residual_norm_relative: float
    matrix_rank: int
    expected_rank: int
    rank_deficient: bool
    backward_error: float
    relative_backward_error: float
    singular_values: SingularValues
    
    @classmethod
    def from_solver(cls, vandermonde: FloatArray, coefficients: FloatArray, 
                    y_data: FloatArray, rank: int, cond: float,
                    singular_values: FloatArray) -> NumericalDiagnostics:
        """Create diagnostics from solver data."""
        residual = vandermonde @ coefficients - y_data
        
        residual_norm_l1 = np.linalg.norm(residual, ord=1)
        residual_norm_l2 = np.linalg.norm(residual, ord=2)
        residual_norm_linf = np.linalg.norm(residual, ord=np.inf)
        
        y_norm = np.linalg.norm(y_data) + MACHINE_EPSILON
        residual_norm_relative = residual_norm_l2 / y_norm
        
        v_norm = np.linalg.norm(vandermonde, ord=2)
        coeff_norm = np.linalg.norm(coefficients)
        denominator = v_norm * coeff_norm + y_norm
        backward_error = residual_norm_l2 / denominator if denominator > 0 else 0.0
        
        expected_rank = vandermonde.shape[1]
        rank_deficient = rank < expected_rank
        digits_lost = np.log10(max(cond, 1.0))
        sv_diag = SingularValues.from_svd(singular_values)
        
        return cls(
            condition_number=cond,
            condition_number_log10=np.log10(max(cond, 1.0)),
            digits_lost=digits_lost,
            residual_norm_l1=float(residual_norm_l1),
            residual_norm_l2=float(residual_norm_l2),
            residual_norm_linf=float(residual_norm_linf),
            residual_norm_relative=float(residual_norm_relative),
            matrix_rank=rank,
            expected_rank=expected_rank,
            rank_deficient=rank_deficient,
            backward_error=float(backward_error),
            relative_backward_error=float(backward_error / (MACHINE_EPSILON + 1.0)),
            singular_values=sv_diag
        )

@dataclass(slots=True)
class VerificationMetrics:
    """Verification metrics comparing fitted polynomial to original data."""
    predictions: FloatArray
    errors: FloatArray
    max_absolute_error: float
    mean_absolute_error: float
    rmse: float
    r_squared: float
    relative_error_percent: float
    
    @classmethod
    def from_data(cls, y_original: FloatArray, y_predicted: FloatArray) -> VerificationMetrics:
        """Create verification metrics from original and predicted values."""
        errors = y_predicted - y_original
        max_error = np.max(np.abs(errors))
        
        sst = np.sum((y_original - np.mean(y_original))**2)
        sse = np.sum(errors**2)
        
        if np.isclose(sst, 0.0, atol=MACHINE_EPSILON):
            r_squared = 1.0
        else:
            r_squared = 1 - sse / sst
        
        relative_error = np.mean(np.abs(errors) / (np.abs(y_original) + MACHINE_EPSILON)) * 100
        
        return cls(
            predictions=y_predicted,
            errors=errors,
            max_absolute_error=float(max_error),
            mean_absolute_error=float(np.mean(np.abs(errors))),
            rmse=float(np.sqrt(np.mean(errors**2))),
            r_squared=float(r_squared),
            relative_error_percent=float(relative_error)
        )

@dataclass(slots=True)
class InterpolationResult:
    """Complete interpolation result."""
    status: SolverStatus
    shifted_coefficients: Coefficients
    expanded_coefficients: Optional[Coefficients]
    shift_value: float
    scale_value: float
    x_data: FloatArray
    y_data: FloatArray
    shifted_x: FloatArray
    degree: int
    method: str
    diagnostics: Optional[NumericalDiagnostics]
    verification: Optional[VerificationMetrics]
    predicted_y: Optional[FloatArray]
    error_message: Optional[str] = None
    
    @property
    def is_success(self) -> bool:
        return self.status == SolverStatus.SUCCESS
    
    @property
    def has_expanded(self) -> bool:
        return self.expanded_coefficients is not None

@dataclass(slots=True)
class ExtrapolationResult:
    """Extrapolation result with diagnostics."""
    x_target: float
    y_predicted: float
    method_used: str
    method_predictions: Dict[str, float]
    reliability_score: float
    reliability_level: ReliabilityLevel
    extrapolation_distance: float
    confidence_interval: Tuple[float, float]
    warning: Optional[str] = None
    execution_time: float = 0.0

# ===================================================================
# EXTRAPOLATION METHODS
# ===================================================================

class LinearExtrapolator:
    """Linear extrapolation using last two points."""
    
    def __init__(self):
        self.slope = 0.0
        self.intercept = 0.0
        self._fitted = False
    
    def fit(self, x_data: FloatArray, y_data: FloatArray) -> None:
        """Fit linear model."""
        if len(x_data) >= 2:
            x1, y1 = x_data[-2], y_data[-2]
            x2, y2 = x_data[-1], y_data[-1]
            dx = x2 - x1
            if abs(dx) > MACHINE_EPSILON:
                self.slope = (y2 - y1) / dx
                self.intercept = y2 - self.slope * x2
            else:
                self.slope = 0.0
                self.intercept = y2
            self._fitted = True
    
    def predict(self, x_target: float) -> float:
        """Predict at target point."""
        if not self._fitted:
            return 0.0
        return self.slope * x_target + self.intercept

class RationalExtrapolator:
    """Rational function extrapolation: y = (p1*x + p2) / (x + p3)."""
    
    def __init__(self):
        self.p1 = 0.0
        self.p2 = 0.0
        self.p3 = 1.0
        self._fitted = False
    
    def fit(self, x_data: FloatArray, y_data: FloatArray) -> None:
        """Fit rational model."""
        if len(x_data) >= 4:
            try:
                x1, y1 = x_data[0], y_data[0]
                x2, y2 = x_data[-1], y_data[-1]
                mid = len(x_data) // 2
                x3, y3 = x_data[mid], y_data[mid]
                
                A = np.array([
                    [x1, 1, -y1],
                    [x2, 1, -y2],
                    [x3, 1, -y3]
                ])
                b = np.array([y1*x1, y2*x2, y3*x3])
                self.p1, self.p2, self.p3 = np.linalg.solve(A, b)
                self._fitted = True
            except:
                self._fitted = False
    
    def predict(self, x_target: float) -> float:
        """Predict at target point."""
        if not self._fitted:
            return 0.0
        denom = x_target + self.p3
        if abs(denom) < MACHINE_EPSILON:
            return 1e12
        return (self.p1 * x_target + self.p2) / denom

class EnsembleExtrapolator:
    """Ensemble of extrapolation methods."""
    
    def __init__(self):
        self.linear = LinearExtrapolator()
        self.rational = RationalExtrapolator()
        self._fitted = False
    
    def fit(self, x_data: FloatArray, y_data: FloatArray) -> None:
        """Fit all methods."""
        self.linear.fit(x_data, y_data)
        self.rational.fit(x_data, y_data)
        self._fitted = True
    
    def predict(self, x_target: float) -> Dict[str, float]:
        """Get predictions from all methods."""
        return {
            'linear': self.linear.predict(x_target),
            'rational': self.rational.predict(x_target)
        }

# ===================================================================
# EXTRAPOLATION ENGINE
# ===================================================================

class ExtrapolationEngine:
    """
    Extrapolation engine with polynomial capping and adaptive blending.
    
    This engine prevents polynomial explosion by capping extreme values
    and blends multiple methods based on extrapolation distance.
    """
    
    def __init__(self, interpolator: 'ShiftedVandermondeInterpolator'):
        self.interpolator = interpolator
        self._x_data = None
        self._y_data = None
        self._x_min = None
        self._x_max = None
        self._x_range = None
        self._linear = LinearExtrapolator()
        self._rational = RationalExtrapolator()
        self._ensemble = EnsembleExtrapolator()
        self._fitted = False
        self._y_mean = 0.0
        self._y_std = 0.0
    
    def fit(self, x_data: FloatArray, y_data: FloatArray) -> None:
        """Fit extrapolation models."""
        x_data = np.asarray(x_data, dtype=np.float64).flatten()
        y_data = np.asarray(y_data, dtype=np.float64).flatten()
        
        self._x_data = x_data
        self._y_data = y_data
        self._x_min = np.min(x_data)
        self._x_max = np.max(x_data)
        self._x_range = self._x_max - self._x_min
        self._y_mean = np.mean(y_data)
        self._y_std = np.std(y_data)
        
        self._linear.fit(x_data, y_data)
        self._rational.fit(x_data, y_data)
        self._ensemble.fit(x_data, y_data)
        self._fitted = True
    
    def _cap_polynomial(self, poly_val: float) -> float:
        """Cap polynomial value to prevent numerical explosion."""
        if self._y_std < MACHINE_EPSILON:
            return self._y_mean
        
        deviation = abs(poly_val - self._y_mean) / (self._y_std + MACHINE_EPSILON)
        if deviation > 5.0:
            capped = self._y_mean + np.sign(poly_val - self._y_mean) * 3.0 * self._y_std
            logger.debug(f"Polynomial capped: {poly_val:.2f} -> {capped:.2f}")
            return capped
        return poly_val
    
    def predict(self, x_target: float) -> ExtrapolationResult:
        """
        Predict at target point with adaptive method selection.
        
        Parameters
        ----------
        x_target : float
            Target x value for prediction
            
        Returns
        -------
        ExtrapolationResult
            Complete extrapolation result with diagnostics
        """
        if not self._fitted:
            raise ValueError("Extrapolation engine not fitted")
        
        start_time = time.time()
        
        # Check if interpolation
        is_extrapolation = (x_target < self._x_min or x_target > self._x_max)
        
        if not is_extrapolation:
            try:
                y_pred = self.interpolator.evaluate(np.array([x_target]))[0]
                return ExtrapolationResult(
                    x_target=x_target,
                    y_predicted=float(y_pred),
                    method_used="interpolation",
                    method_predictions={'interpolation': float(y_pred)},
                    reliability_score=1.0,
                    reliability_level=ReliabilityLevel.HIGH,
                    extrapolation_distance=0.0,
                    confidence_interval=(y_pred - 0.01*abs(y_pred), y_pred + 0.01*abs(y_pred)),
                    warning=None,
                    execution_time=time.time() - start_time
                )
            except:
                pass
        
        # Calculate extrapolation distance
        if x_target < self._x_min:
            distance = (self._x_min - x_target) / (self._x_range + MACHINE_EPSILON)
        else:
            distance = (x_target - self._x_max) / (self._x_range + MACHINE_EPSILON)
        
        # Get predictions from all methods
        predictions = {}
        
        # Polynomial with capping
        try:
            poly_val = float(self.interpolator.evaluate(
                np.array([x_target]), use_shifted=True
            )[0])
            predictions['polynomial'] = self._cap_polynomial(poly_val)
        except:
            predictions['polynomial'] = None
        
        # Linear
        try:
            predictions['linear'] = self._linear.predict(x_target)
        except:
            predictions['linear'] = None
        
        # Rational
        try:
            predictions['rational'] = self._rational.predict(x_target)
        except:
            predictions['rational'] = None
        
        # Adaptive method selection based on distance
        if distance < 0.1:
            # Close extrapolation: polynomial with linear blend
            y_poly = predictions['polynomial'] if predictions['polynomial'] is not None else 0
            y_linear = predictions['linear'] if predictions['linear'] is not None else 0
            y_pred = 0.7 * y_poly + 0.3 * y_linear
            method_used = 'blended'
            reliability = 0.85
            
        elif distance < 0.5:
            # Moderate extrapolation: ensemble with reduced polynomial weight
            valid_preds = []
            weights = []
            for name, p in predictions.items():
                if p is not None:
                    valid_preds.append(p)
                    if name == 'polynomial':
                        weights.append(0.25)
                    elif name == 'linear':
                        weights.append(0.45)
                    else:
                        weights.append(0.30)
            
            if valid_preds:
                weights = np.array(weights) / np.sum(weights)
                y_pred = np.average(valid_preds, weights=weights)
                method_used = 'ensemble'
                reliability = 0.65
            else:
                y_pred = predictions['linear'] or 0
                method_used = 'linear'
                reliability = 0.5
                
        else:
            # Large extrapolation: linear with rational influence
            y_linear = predictions['linear'] if predictions['linear'] is not None else 0
            y_rational = predictions['rational'] if predictions['rational'] is not None else 0
            y_pred = 0.7 * y_linear + 0.3 * y_rational
            method_used = 'linear_rational'
            reliability = max(0.1, 0.7 - distance * 0.04)
        
        # Determine reliability level
        if reliability >= 0.8:
            rel_level = ReliabilityLevel.HIGH
        elif reliability >= 0.5:
            rel_level = ReliabilityLevel.MEDIUM
        elif reliability >= 0.2:
            rel_level = ReliabilityLevel.LOW
        else:
            rel_level = ReliabilityLevel.VERY_LOW
        
        # Generate warning
        warning = None
        if distance > 2.0:
            warning = f"Large extrapolation: {distance:.1f}x beyond data range"
        elif distance > 0.5 and reliability < 0.4:
            warning = f"Extrapolation: {distance:.1f}x beyond data range"
        elif reliability < 0.3:
            warning = "Low reliability - predictions may be inaccurate"
        
        # Calculate confidence interval
        valid_preds = [p for p in predictions.values() if p is not None]
        if len(valid_preds) > 1:
            std_pred = np.std(valid_preds)
            confidence_width = 1.96 * std_pred * (1 + distance * 0.3)
            confidence_interval = (y_pred - confidence_width, y_pred + confidence_width)
        else:
            uncertainty = 0.15 * abs(y_pred) * (1 + distance * 0.2)
            confidence_interval = (y_pred - uncertainty, y_pred + uncertainty)
        
        return ExtrapolationResult(
            x_target=x_target,
            y_predicted=float(y_pred),
            method_used=method_used,
            method_predictions={k: float(v) if v is not None else None 
                              for k, v in predictions.items()},
            reliability_score=float(reliability),
            reliability_level=rel_level,
            extrapolation_distance=float(distance),
            confidence_interval=(float(confidence_interval[0]), 
                               float(confidence_interval[1])),
            warning=warning,
            execution_time=time.time() - start_time
        )

# ===================================================================
# VISUALIZATION ENGINE
# ===================================================================

class VisualizationEngine:
    """
    Professional plotting engine for interpolation and extrapolation.
    
    Creates publication-quality figures with Matplotlib showing:
    - Data points with error bars
    - Interpolating polynomial
    - Extrapolation with confidence bands
    - Multiple method comparison
    - Residual plots
    """
    
    def __init__(self):
        self.figsize = (10, 6)
        self.dpi = 120
        self.point_color = '#0d47a1'
        self.point_size = 80
        self.line_color = '#1565c0'
        self.extrap_color = '#e65100'
        self.band_color = '#ffccbc'
        self.residual_color = '#d32f2f'
        self.grid_alpha = 0.3
        
    def plot_interpolation(self, result: InterpolationResult, 
                           x_range: Optional[Tuple[float, float]] = None,
                           title: str = "Polynomial Interpolation") -> plt.Figure:
        """
        Create a publication-quality interpolation plot.
        """
        if not result.is_success:
            raise ValueError("Cannot plot failed interpolation result")
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # Determine x range for plotting
        if x_range is None:
            x_min = np.min(result.x_data) - 0.15 * np.ptp(result.x_data)
            x_max = np.max(result.x_data) + 0.15 * np.ptp(result.x_data)
        else:
            x_min, x_max = x_range
        
        x_plot = np.linspace(x_min, x_max, PLOT_POINTS)
        
        # Evaluate polynomial
        try:
            y_plot = result.x_data * 0  # Placeholder
            # Use interpolator for evaluation
            from . import ShiftedVandermondeInterpolator
            temp_interp = ShiftedVandermondeInterpolator()
            temp_interp._last_result = result
            y_plot = temp_interp.evaluate(x_plot, use_shifted=True)
        except:
            # Fallback: use coefficients directly
            coeffs = result.expanded_coefficients if result.expanded_coefficients is not None else result.shifted_coefficients
            y_plot = np.polyval(coeffs[::-1], x_plot)
        
        # Plot 1: Main interpolation plot
        ax1.scatter(result.x_data, result.y_data, 
                   color=self.point_color, s=self.point_size,
                   zorder=3, label='Data Points', edgecolors='white', linewidth=1.5)
        ax1.plot(x_plot, y_plot, color=self.line_color, linewidth=2.5,
                label=f'Polynomial (degree {result.degree})', zorder=2)
        
        # Add data point labels
        for i, (x, y) in enumerate(zip(result.x_data, result.y_data)):
            ax1.annotate(f'({x:.3f}, {y:.4f})', 
                        (x, y), 
                        xytext=(5, 5), 
                        textcoords='offset points',
                        fontsize=8, alpha=0.7)
        
        ax1.set_xlabel('X', fontsize=11)
        ax1.set_ylabel('Y', fontsize=11)
        ax1.set_title(title, fontsize=12)
        ax1.grid(True, alpha=self.grid_alpha)
        ax1.legend(loc='best')
        
        # Plot 2: Residual plot
        residuals = result.y_data - result.predicted_y
        ax2.scatter(result.x_data, residuals, 
                   color=self.residual_color, s=self.point_size,
                   zorder=3, label='Residuals', edgecolors='white', linewidth=1.5)
        ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        
        # Add zero line
        ax2.set_xlabel('X', fontsize=11)
        ax2.set_ylabel('Residual', fontsize=11)
        ax2.set_title('Residual Analysis', fontsize=12)
        ax2.grid(True, alpha=self.grid_alpha)
        ax2.legend(loc='best')
        
        # Add residual statistics
        max_res = np.max(np.abs(residuals))
        rmse = np.sqrt(np.mean(residuals**2))
        stats_text = f'Max Residual: {max_res:.2e}\nRMSE: {rmse:.2e}'
        ax2.text(0.02, 0.98, stats_text, transform=ax2.transAxes,
                verticalalignment='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        plt.tight_layout()
        return fig
    
    def plot_extrapolation(self, result: ExtrapolationResult, 
                           interpolation_result: InterpolationResult,
                           x_range: Optional[Tuple[float, float]] = None,
                           title: str = "Extrapolation Analysis") -> plt.Figure:
        """
        Create a publication-quality extrapolation plot with confidence bands.
        """
        if not interpolation_result.is_success:
            raise ValueError("Cannot plot without interpolation result")
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        ax1, ax2, ax3, ax4 = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]
        
        # Determine x range
        if x_range is None:
            x_min = min(interpolation_result.x_data) - 0.5 * np.ptp(interpolation_result.x_data)
            x_max = max(interpolation_result.x_data) + 0.5 * np.ptp(interpolation_result.x_data)
            # Ensure target is included
            if result.x_target < x_min:
                x_min = result.x_target - 0.1 * abs(result.x_target)
            if result.x_target > x_max:
                x_max = result.x_target + 0.1 * abs(result.x_target)
        else:
            x_min, x_max = x_range
        
        x_plot = np.linspace(x_min, x_max, PLOT_POINTS)
        
        # Evaluate polynomial
        try:
            from . import ShiftedVandermondeInterpolator
            temp_interp = ShiftedVandermondeInterpolator()
            temp_interp._last_result = interpolation_result
            y_plot = temp_interp.evaluate(x_plot, use_shifted=True)
        except:
            coeffs = interpolation_result.expanded_coefficients if interpolation_result.expanded_coefficients is not None else interpolation_result.shifted_coefficients
            y_plot = np.polyval(coeffs[::-1], x_plot)
        
        # ============================================================
        # Plot 1: Main interpolation + extrapolation
        # ============================================================
        
        # Split into interpolation and extrapolation regions
        x_min_data = np.min(interpolation_result.x_data)
        x_max_data = np.max(interpolation_result.x_data)
        
        interp_mask = (x_plot >= x_min_data) & (x_plot <= x_max_data)
        extrap_mask = (x_plot < x_min_data) | (x_plot > x_max_data)
        
        # Data points
        ax1.scatter(interpolation_result.x_data, interpolation_result.y_data,
                   color=self.point_color, s=self.point_size,
                   zorder=3, label='Data Points', edgecolors='white', linewidth=1.5)
        
        # Interpolation region (solid line)
        if np.any(interp_mask):
            ax1.plot(x_plot[interp_mask], y_plot[interp_mask],
                    color=self.line_color, linewidth=2.5,
                    label='Interpolation', zorder=2)
        
        # Extrapolation region (dashed line)
        if np.any(extrap_mask):
            ax1.plot(x_plot[extrap_mask], y_plot[extrap_mask],
                    color=self.extrap_color, linewidth=2.0, linestyle='--',
                    label='Polynomial Extrapolation', zorder=1, alpha=0.7)
        
        # Target point
        ax1.scatter(result.x_target, result.y_predicted,
                   color=self.extrap_color, s=120, marker='D',
                   zorder=4, label=f'Prediction: {result.y_predicted:.4f}',
                   edgecolors='black', linewidth=2)
        
        # Confidence interval
        ci_low, ci_high = result.confidence_interval
        ax1.axhline(y=ci_low, color='red', linestyle=':', linewidth=1.5, alpha=0.5)
        ax1.axhline(y=ci_high, color='red', linestyle=':', linewidth=1.5, alpha=0.5)
        ax1.fill_between([result.x_target - 0.3, result.x_target + 0.3],
                        [ci_low, ci_low], [ci_high, ci_high],
                        color='red', alpha=0.15, label='95% Confidence Interval')
        
        # Vertical line at target
        ax1.axvline(x=result.x_target, color='gray', linestyle='--', 
                   linewidth=1, alpha=0.5)
        
        # Data range indicator
        ax1.axvspan(x_min_data, x_max_data, alpha=0.08, color='blue',
                   label='Data Range')
        
        ax1.set_xlabel('X', fontsize=11)
        ax1.set_ylabel('Y', fontsize=11)
        ax1.set_title('Interpolation & Extrapolation', fontsize=12)
        ax1.grid(True, alpha=self.grid_alpha)
        ax1.legend(loc='best', fontsize=8)
        
        # ============================================================
        # Plot 2: Method Comparison
        # ============================================================
        
        methods = list(result.method_predictions.keys())
        values = [v for v in result.method_predictions.values() if v is not None]
        valid_methods = [m for m, v in zip(methods, result.method_predictions.values()) if v is not None]
        
        if valid_methods:
            bars = ax2.bar(valid_methods, values,
                          color=['#1565c0', '#e65100', '#2e7d32', '#6a1b9a', '#f57c00'][:len(valid_methods)],
                          alpha=0.7, edgecolor='black', linewidth=1)
            
            # Add value labels on bars
            for bar, val in zip(bars, values):
                ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05*max(values),
                        f'{val:.2f}', ha='center', va='bottom', fontsize=9)
            
            # Highlight selected method
            for i, (method, bar) in enumerate(zip(valid_methods, bars)):
                if method == result.method_used:
                    bar.set_edgecolor('black')
                    bar.set_linewidth(3)
                    bar.set_alpha(0.9)
        
        ax2.set_xlabel('Method', fontsize=11)
        ax2.set_ylabel('Predicted Value', fontsize=11)
        ax2.set_title(f'Method Comparison at X = {result.x_target:.4f}', fontsize=12)
        ax2.grid(True, alpha=self.grid_alpha, axis='y')
        ax2.tick_params(axis='x', rotation=15)
        
        # ============================================================
        # Plot 3: Reliability Gauge
        # ============================================================
        
        # Create a simple gauge
        reliability = result.reliability_score
        colors_gauge = ['#d32f2f', '#f57c00', '#ffeb3b', '#4caf50', '#2e7d32']
        
        # Circular gauge
        theta = np.linspace(0, np.pi, 100)
        r = 0.8
        
        # Background arc
        ax3.plot(np.cos(theta), np.sin(theta), color='lightgray', linewidth=20, alpha=0.3)
        
        # Reliability arc
        reliability_angle = reliability * np.pi
        theta_rel = np.linspace(0, reliability_angle, 50)
        ax3.plot(np.cos(theta_rel), np.sin(theta_rel), 
                color=colors_gauge[int(reliability * 4)], linewidth=20, alpha=0.8)
        
        # Add tick marks
        for angle in [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]:
            x_tick = np.cos(angle) * 1.05
            y_tick = np.sin(angle) * 1.05
            ax3.text(x_tick, y_tick, f'{angle/np.pi*100:.0f}%', 
                    ha='center', va='center', fontsize=8, alpha=0.7)
        
        # Center text
        ax3.text(0, 0.2, f'{reliability*100:.1f}%', 
                ha='center', va='center', fontsize=24, fontweight='bold')
        ax3.text(0, -0.2, result.reliability_level.value.upper(), 
                ha='center', va='center', fontsize=12, alpha=0.7)
        
        # Distance indicator
        ax3.text(0, -0.5, f'Distance: {result.extrapolation_distance:.2f}x', 
                ha='center', va='center', fontsize=10, alpha=0.7)
        
        ax3.set_xlim(-1.2, 1.2)
        ax3.set_ylim(-0.8, 1.2)
        ax3.set_aspect('equal')
        ax3.axis('off')
        ax3.set_title('Reliability Assessment', fontsize=12)
        
        # ============================================================
        # Plot 4: Method Predictions Comparison
        # ============================================================
        
        # Create a comparison plot showing all method predictions
        if valid_methods:
            x_pos = np.arange(len(valid_methods))
            width = 0.35
            
            # Show each method's prediction with error bars
            means = values
            stds = [abs(v - result.y_predicted) for v in values]
            
            ax4.bar(x_pos, means, width, color=['#1565c0', '#e65100', '#2e7d32', '#6a1b9a'][:len(valid_methods)],
                   alpha=0.7, yerr=stds, capsize=5, edgecolor='black', linewidth=1)
            
            # Highlight selected method
            for i, (method, bar) in enumerate(zip(valid_methods, ax4.patches)):
                if method == result.method_used:
                    bar.set_edgecolor('black')
                    bar.set_linewidth(3)
            
            # Add selected method label
            ax4.axhline(y=result.y_predicted, color='red', linestyle='--', 
                       linewidth=1.5, alpha=0.7, label=f'Selected: {result.y_predicted:.2f}')
            
            ax4.set_xticks(x_pos)
            ax4.set_xticklabels(valid_methods, rotation=15, ha='right')
            ax4.set_ylabel('Predicted Value', fontsize=11)
            ax4.set_title('Method Comparison with Uncertainty', fontsize=12)
            ax4.grid(True, alpha=self.grid_alpha, axis='y')
            ax4.legend(loc='best', fontsize=8)
        
        plt.tight_layout()
        return fig
    
    def plot_comprehensive(self, interpolation_result: InterpolationResult,
                          extrapolation_results: List[ExtrapolationResult],
                          title: str = "Comprehensive Analysis") -> plt.Figure:
        """
        Create a comprehensive plot combining all results.
        """
        if not interpolation_result.is_success:
            raise ValueError("Cannot plot without interpolation result")
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        ax1, ax2, ax3, ax4 = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]
        
        # Get data range
        x_min_data = np.min(interpolation_result.x_data)
        x_max_data = np.max(interpolation_result.x_data)
        y_min_data = np.min(interpolation_result.y_data)
        y_max_data = np.max(interpolation_result.y_data)
        
        # Determine x range for plotting
        x_min_plot = x_min_data - 0.3 * np.ptp(interpolation_result.x_data)
        x_max_plot = x_max_data + 0.3 * np.ptp(interpolation_result.x_data)
        
        # Include extrapolation targets
        for res in extrapolation_results:
            if res.x_target < x_min_plot:
                x_min_plot = res.x_target - 0.1 * abs(res.x_target)
            if res.x_target > x_max_plot:
                x_max_plot = res.x_target + 0.1 * abs(res.x_target)
        
        x_plot = np.linspace(x_min_plot, x_max_plot, PLOT_POINTS)
        
        # Evaluate polynomial
        try:
            from . import ShiftedVandermondeInterpolator
            temp_interp = ShiftedVandermondeInterpolator()
            temp_interp._last_result = interpolation_result
            y_plot = temp_interp.evaluate(x_plot, use_shifted=True)
        except:
            coeffs = interpolation_result.expanded_coefficients if interpolation_result.expanded_coefficients is not None else interpolation_result.shifted_coefficients
            y_plot = np.polyval(coeffs[::-1], x_plot)
        
        # ============================================================
        # Plot 1: Main plot with all predictions
        # ============================================================
        
        # Data points
        ax1.scatter(interpolation_result.x_data, interpolation_result.y_data,
                   color=self.point_color, s=self.point_size,
                   zorder=3, label='Data', edgecolors='white', linewidth=1.5)
        
        # Interpolation region
        interp_mask = (x_plot >= x_min_data) & (x_plot <= x_max_data)
        if np.any(interp_mask):
            ax1.plot(x_plot[interp_mask], y_plot[interp_mask],
                    color=self.line_color, linewidth=2.5,
                    label='Interpolation', zorder=2)
        
        # Extrapolation region
        extrap_mask = (x_plot < x_min_data) | (x_plot > x_max_data)
        if np.any(extrap_mask):
            ax1.plot(x_plot[extrap_mask], y_plot[extrap_mask],
                    color=self.extrap_color, linewidth=2.0, linestyle='--',
                    label='Polynomial Extrapolation', zorder=1, alpha=0.7)
        
        # Data range indicator
        ax1.axvspan(x_min_data, x_max_data, alpha=0.06, color='blue')
        
        # Extrapolation predictions
        for res in extrapolation_results:
            ax1.scatter(res.x_target, res.y_predicted,
                       color=self.extrap_color, s=100, marker='D',
                       zorder=4, edgecolors='black', linewidth=2)
            
            # Confidence interval
            ci_low, ci_high = res.confidence_interval
            ax1.errorbar(res.x_target, res.y_predicted,
                        yerr=[[res.y_predicted - ci_low], [ci_high - res.y_predicted]],
                        color='red', fmt='none', capsize=5, alpha=0.5, zorder=3)
        
        ax1.set_xlabel('X', fontsize=11)
        ax1.set_ylabel('Y', fontsize=11)
        ax1.set_title('Interpolation and Extrapolation', fontsize=12)
        ax1.grid(True, alpha=self.grid_alpha)
        ax1.legend(loc='best', fontsize=8)
        
        # ============================================================
        # Plot 2: Residuals
        # ============================================================
        
        residuals = interpolation_result.y_data - interpolation_result.predicted_y
        ax2.scatter(interpolation_result.x_data, residuals,
                   color=self.residual_color, s=self.point_size,
                   zorder=3, label='Residuals', edgecolors='white', linewidth=1.5)
        ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        
        # Add residual statistics
        max_res = np.max(np.abs(residuals))
        rmse = np.sqrt(np.mean(residuals**2))
        ax2.text(0.02, 0.98, f'Max: {max_res:.2e}\nRMSE: {rmse:.2e}',
                transform=ax2.transAxes, verticalalignment='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        ax2.set_xlabel('X', fontsize=11)
        ax2.set_ylabel('Residual', fontsize=11)
        ax2.set_title('Residual Analysis', fontsize=12)
        ax2.grid(True, alpha=self.grid_alpha)
        ax2.legend(loc='best')
        
        # ============================================================
        # Plot 3: Reliability vs Distance
        # ============================================================
        
        if extrapolation_results:
            distances = [res.extrapolation_distance for res in extrapolation_results]
            reliabilities = [res.reliability_score for res in extrapolation_results]
            
            ax3.scatter(distances, reliabilities, s=80, c='#0d47a1',
                       zorder=3, label='Predictions', edgecolors='white', linewidth=1.5)
            
            # Add labels
            for res in extrapolation_results:
                ax3.annotate(f'{res.x_target:.2f}', 
                            (res.extrapolation_distance, res.reliability_score),
                            xytext=(5, 5), textcoords='offset points',
                            fontsize=8, alpha=0.7)
            
            # Theoretical reliability curve
            d_range = np.linspace(0, max(distances) * 1.2, 100)
            reliability_curve = np.maximum(0.1, 0.7 - d_range * 0.04)
            ax3.plot(d_range, reliability_curve, color='gray', linestyle='--',
                    alpha=0.5, label='Theoretical Decay')
            
            ax3.set_xlabel('Extrapolation Distance (x)', fontsize=11)
            ax3.set_ylabel('Reliability Score', fontsize=11)
            ax3.set_title('Reliability vs Extrapolation Distance', fontsize=12)
            ax3.set_ylim(0, 1.05)
            ax3.grid(True, alpha=self.grid_alpha)
            ax3.legend(loc='best')
        
        # ============================================================
        # Plot 4: Method Performance Summary
        # ============================================================
        
        if extrapolation_results:
            # Collect all method names
            all_methods = set()
            for res in extrapolation_results:
                all_methods.update(res.method_predictions.keys())
            all_methods = sorted([m for m in all_methods if m is not None])
            
            # Create comparison table
            if all_methods:
                data = []
                for res in extrapolation_results:
                    row = [res.x_target]
                    for method in all_methods:
                        if method in res.method_predictions and res.method_predictions[method] is not None:
                            row.append(res.method_predictions[method])
                        else:
                            row.append(np.nan)
                    data.append(row)
                
                data_array = np.array(data)
                
                # Plot as a table
                ax4.axis('off')
                
                # Create table
                col_labels = ['X'] + all_methods + ['Selected']
                
                table_data = []
                for i, res in enumerate(extrapolation_results):
                    row = [f'{res.x_target:.4f}']
                    for method in all_methods:
                        if method in res.method_predictions and res.method_predictions[method] is not None:
                            row.append(f'{res.method_predictions[method]:.4f}')
                        else:
                            row.append('N/A')
                    row.append(f'{res.y_predicted:.4f}')
                    table_data.append(row)
                
                # Color coding for table
                colors = [['#f8f9fa'] * len(col_labels) for _ in range(len(table_data))]
                
                table = ax4.table(cellText=table_data, colLabels=col_labels,
                                 cellLoc='center', loc='center',
                                 cellColours=colors,
                                 colColours=['#0d47a1'] * len(col_labels))
                table.auto_set_font_size(False)
                table.set_fontsize(9)
                table.scale(1, 1.5)
                
                ax4.set_title('Extrapolation Results Summary', fontsize=12, pad=20)
        
        plt.tight_layout()
        return fig

# ===================================================================
# CORE SOLVER
# ===================================================================

class ShiftedVandermondeInterpolator:
    """
    Polynomial interpolation using shifted and scaled Vandermonde basis.
    
    Implements both exact interpolation and least-squares fitting with
    comprehensive numerical diagnostics.
    """
    
    def __init__(self, shift_method: ShiftMethod = ShiftMethod.MEAN, 
                 scale_data: bool = True,
                 expansion_mode: ExpansionMode = ExpansionMode.BOTH):
        self.shift_method = shift_method
        self.scale_data = scale_data
        self.expansion_mode = expansion_mode
        self._last_result: Optional[InterpolationResult] = None
        self._cached_expanded: Optional[Coefficients] = None
        self._cached_predictions: Optional[FloatArray] = None
        self._cached_verification: Optional[VerificationMetrics] = None
        self._cached_diagnostics: Optional[NumericalDiagnostics] = None
        self._extrapolation_engine = None
        self._visualization = VisualizationEngine()
    
    def interpolate(self, x_data: FloatArray, y_data: FloatArray) -> InterpolationResult:
        """Exact polynomial interpolation through all data points."""
        return self._solve(x_data, y_data, use_lstsq=False)
    
    def fit(self, x_data: FloatArray, y_data: FloatArray, 
            degree: Optional[int] = None) -> InterpolationResult:
        """Least-squares polynomial fitting."""
        return self._solve(x_data, y_data, use_lstsq=True, degree=degree)
    
    def _solve(self, x_data: FloatArray, y_data: FloatArray,
               use_lstsq: bool = False, degree: Optional[int] = None) -> InterpolationResult:
        """Core solver implementation."""
        try:
            x_data = np.asarray(x_data, dtype=np.float64).flatten()
            y_data = np.asarray(y_data, dtype=np.float64).flatten()
            
            if len(x_data) != len(y_data):
                return self._error_result(f"Array length mismatch: {len(x_data)} vs {len(y_data)}")
            
            if len(x_data) < 2:
                return self._error_result("At least 2 points required")
            
            if np.any(~np.isfinite(x_data)) or np.any(~np.isfinite(y_data)):
                return self._error_result("NaN or Inf values detected")
            
            if not use_lstsq:
                unique_x = np.unique(x_data)
                if len(unique_x) < len(x_data):
                    return self._error_result("Duplicate x values detected")
            
            # Apply shift
            if self.shift_method == ShiftMethod.MEAN:
                shift = np.mean(x_data)
            elif self.shift_method == ShiftMethod.FIRST:
                shift = x_data[0]
            else:
                shift = 0.0
            
            # Apply adaptive scaling
            if self.scale_data:
                std_dev = np.std(x_data)
                if std_dev > MACHINE_EPSILON:
                    scale = 1.0 / std_dev
                else:
                    x_range = np.max(x_data) - np.min(x_data)
                    scale = 1.0 / (x_range + MACHINE_EPSILON)
                shifted_x = (x_data - shift) * scale
            else:
                scale = 1.0
                shifted_x = x_data - shift
            
            # Determine degree
            if use_lstsq:
                if degree is None:
                    degree = min(DEFAULT_DEGREE, len(x_data) - 1)
                elif degree >= len(x_data):
                    degree = len(x_data) - 1
                degree = max(1, degree)
            else:
                degree = len(x_data) - 1
            
            # Build Vandermonde matrix
            vandermonde = np.vander(shifted_x, N=degree + 1, increasing=True)
            matrix_rank = np.linalg.matrix_rank(vandermonde)
            
            # Solve system
            if use_lstsq and len(x_data) > degree + 1:
                try:
                    u, s, vt = np.linalg.svd(vandermonde, full_matrices=False)
                    s_inv = np.where(s > MACHINE_EPSILON * 1e3, 1.0 / s, 0.0)
                    coefficients = vt.T @ (s_inv * (u.T @ y_data))
                    singular_values = s
                    method = "least_squares"
                except np.linalg.LinAlgError as e:
                    return self._error_result(f"Least-squares failed: {str(e)}")
            else:
                try:
                    coefficients = np.linalg.solve(vandermonde, y_data)
                    _, singular_values, _ = np.linalg.svd(vandermonde)
                    method = "interpolation"
                except np.linalg.LinAlgError as e:
                    return self._error_result(f"Singular matrix: {str(e)}")
            
            coefficients = coefficients.flatten()
            condition_number = np.linalg.cond(vandermonde)
            
            diagnostics = NumericalDiagnostics.from_solver(
                vandermonde, coefficients, y_data, matrix_rank, 
                condition_number, singular_values
            )
            
            predicted_y = np.polyval(coefficients[::-1], shifted_x)
            verification = VerificationMetrics.from_data(y_data, predicted_y)
            
            expanded_coeffs = None
            if self.expansion_mode in [ExpansionMode.EXPANDED, ExpansionMode.BOTH]:
                expanded_coeffs = self._expand_coefficients_exact(coefficients, shift, scale)
                self._cached_expanded = expanded_coeffs
            
            self._cached_predictions = predicted_y
            self._cached_verification = verification
            self._cached_diagnostics = diagnostics
            
            result = InterpolationResult(
                status=SolverStatus.SUCCESS,
                shifted_coefficients=coefficients,
                expanded_coefficients=expanded_coeffs,
                shift_value=shift,
                scale_value=scale,
                x_data=x_data,
                y_data=y_data,
                shifted_x=shifted_x,
                degree=degree,
                method=method,
                diagnostics=diagnostics,
                verification=verification,
                predicted_y=predicted_y,
                error_message=None
            )
            
            self._last_result = result
            
            # Initialize extrapolation engine
            self._extrapolation_engine = ExtrapolationEngine(self)
            self._extrapolation_engine.fit(x_data, y_data)
            
            if condition_number > CONDITION_WARNING:
                logger.warning(f"Condition number: {condition_number:.2e} ({diagnostics.digits_lost:.1f} digits lost)")
            
            if diagnostics.rank_deficient:
                logger.warning(f"Rank deficient: {matrix_rank} < {degree + 1}")
            
            x_range = np.max(x_data) - np.min(x_data)
            if x_range < MACHINE_EPSILON:
                logger.warning("Very small x range - results may be unstable")
            
            return result
            
        except Exception as e:
            logger.error(f"Solver error: {str(e)}")
            return self._error_result(f"Unexpected error: {str(e)}")
    
    def _expand_coefficients_exact(self, shifted_coeffs: Coefficients, 
                                    shift: float, scale: float) -> Coefficients:
        """Exactly expand shifted and scaled coefficients to original basis."""
        degree = len(shifted_coeffs) - 1
        expanded = np.zeros(degree + 1, dtype=np.float64)
        
        for i, c in enumerate(shifted_coeffs):
            if abs(c) < MACHINE_EPSILON:
                continue
            
            scaled_c = c * (scale ** i)
            
            for j in range(i + 1):
                binom = comb(i, j)
                term = scaled_c * binom * ((-shift) ** (i - j))
                expanded[degree - j] += term
        
        return expanded
    
    def evaluate(self, x_values: FloatArray, 
                 use_shifted: bool = True) -> FloatArray:
        """Evaluate polynomial at specified points."""
        if self._last_result is None:
            raise ValueError("No solution available. Run interpolate() or fit() first.")
        
        x_array = np.asarray(x_values, dtype=np.float64).flatten()
        
        x_min = np.min(self._last_result.x_data)
        x_max = np.max(self._last_result.x_data)
        outside = np.any((x_array < x_min) | (x_array > x_max))
        
        if outside:
            logger.info("Evaluation performed outside interpolation interval")
        
        if use_shifted:
            shifted_x = (x_array - self._last_result.shift_value) * self._last_result.scale_value
            coefficients = self._last_result.shifted_coefficients
        else:
            shifted_x = x_array
            coefficients = self._last_result.expanded_coefficients
            
            if coefficients is None:
                coefficients = self._expand_coefficients_exact(
                    self._last_result.shifted_coefficients,
                    self._last_result.shift_value,
                    self._last_result.scale_value
                )
                self._last_result.expanded_coefficients = coefficients
        
        return np.polyval(coefficients[::-1], shifted_x)
    
    def extrapolate(self, x_target: float) -> ExtrapolationResult:
        """
        Extrapolate using adaptive method selection.
        
        Parameters
        ----------
        x_target : float
            Target x value for prediction
            
        Returns
        -------
        ExtrapolationResult
            Complete extrapolation result with diagnostics
        """
        if self._extrapolation_engine is None:
            raise ValueError("No interpolation result available. Run interpolate() or fit() first.")
        
        return self._extrapolation_engine.predict(x_target)
    
    def plot_interpolation(self, x_range: Optional[Tuple[float, float]] = None,
                          title: str = "Polynomial Interpolation") -> plt.Figure:
        """Generate interpolation plot."""
        if self._last_result is None:
            raise ValueError("No interpolation result available")
        return self._visualization.plot_interpolation(self._last_result, x_range, title)
    
    def plot_extrapolation(self, x_target: float,
                          x_range: Optional[Tuple[float, float]] = None,
                          title: str = "Extrapolation Analysis") -> plt.Figure:
        """Generate extrapolation plot."""
        if self._last_result is None:
            raise ValueError("No interpolation result available")
        result = self.extrapolate(x_target)
        return self._visualization.plot_extrapolation(result, self._last_result, x_range, title)
    
    def plot_comprehensive(self, x_targets: List[float],
                          title: str = "Comprehensive Analysis") -> plt.Figure:
        """Generate comprehensive plot with multiple extrapolations."""
        if self._last_result is None:
            raise ValueError("No interpolation result available")
        results = [self.extrapolate(xt) for xt in x_targets]
        return self._visualization.plot_comprehensive(self._last_result, results, title)
    
    def get_expanded(self) -> Optional[Coefficients]:
        """Get expanded coefficients (lazy computation)."""
        if self._last_result is None:
            return None
        
        if self._last_result.expanded_coefficients is not None:
            return self._last_result.expanded_coefficients
        
        expanded = self._expand_coefficients_exact(
            self._last_result.shifted_coefficients,
            self._last_result.shift_value,
            self._last_result.scale_value
        )
        self._last_result.expanded_coefficients = expanded
        return expanded
    
    def _error_result(self, message: str) -> InterpolationResult:
        """Create error result."""
        return InterpolationResult(
            status=SolverStatus.ERROR,
            shifted_coefficients=np.array([]),
            expanded_coefficients=None,
            shift_value=0.0,
            scale_value=1.0,
            x_data=np.array([]),
            y_data=np.array([]),
            shifted_x=np.array([]),
            degree=0,
            method="none",
            diagnostics=None,
            verification=None,
            predicted_y=None,
            error_message=message
        )
    
    @property
    def has_result(self) -> bool:
        return self._last_result is not None and self._last_result.is_success

# ===================================================================
# POLYNOMIAL FORMATTER
# ===================================================================

class PolynomialFormatter:
    """Format polynomials for display and export."""
    
    @staticmethod
    def format_shifted(coeffs: Coefficients, shift: float, scale: float,
                       variable: str = "x", scientific: bool = False) -> str:
        """Format shifted polynomial."""
        if len(coeffs) == 0:
            return "P(x) = 0"
        
        degree = len(coeffs) - 1
        terms = []
        
        x_expr = f"({variable} - {shift:.4f})"
        if scale != 1.0:
            x_expr = f"{scale:.4f} * {x_expr}"
        
        for i, c in enumerate(coeffs):
            power = degree - i
            if abs(c) < 10 * MACHINE_EPSILON:
                continue
            
            c_str = PolynomialFormatter._format_coefficient(c, scientific)
            
            if power == 0:
                terms.append(c_str)
            elif power == 1:
                terms.append(f"{c_str}{x_expr}")
            else:
                terms.append(f"{c_str}{x_expr}^{power}")
        
        if not terms:
            return "P(x) = 0"
        
        return "P(x) = " + " + ".join(terms).replace("+ -", "- ")
    
    @staticmethod
    def format_expanded(coeffs: Coefficients, variable: str = "T", 
                        scientific: bool = False) -> str:
        """Format expanded polynomial."""
        if len(coeffs) == 0:
            return "P(T) = 0"
        
        degree = len(coeffs) - 1
        terms = []
        
        for i, c in enumerate(coeffs):
            power = degree - i
            if abs(c) < 10 * MACHINE_EPSILON:
                continue
            
            c_str = PolynomialFormatter._format_coefficient(c, scientific)
            
            if power == 0:
                terms.append(c_str)
            elif power == 1:
                terms.append(f"{c_str}{variable}")
            else:
                terms.append(f"{c_str}{variable}^{power}")
        
        if not terms:
            return "P(T) = 0"
        
        return "P(T) = " + " + ".join(terms).replace("+ -", "- ")
    
    @staticmethod
    def _format_coefficient(value: float, scientific: bool) -> str:
        """Format coefficient with appropriate precision."""
        if abs(value) < MACHINE_EPSILON:
            return "0"
        
        if scientific or abs(value) >= 1e6 or abs(value) <= 1e-6:
            return f"{value:.6e}"
        
        if abs(value) >= 1:
            return f"{value:.8f}".rstrip('0').rstrip('.')
        else:
            return f"{value:.10f}".rstrip('0').rstrip('.')

# ===================================================================
# WIDGETS
# ===================================================================

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import io
import base64

class DataInputWidget:
    """Enhanced data input widget with import/export."""
    
    def __init__(self, default_x: Optional[FloatArray] = None,
                 default_y: Optional[FloatArray] = None):
        self.x_inputs: List[widgets.FloatText] = []
        self.y_inputs: List[widgets.FloatText] = []
        self.container = widgets.VBox()
        self.num_points = 4
        self.default_x = default_x
        self.default_y = default_y
        self._build_inputs()
    
    def _build_inputs(self):
        self.x_inputs = []
        self.y_inputs = []
        
        header = widgets.HBox([
            widgets.Label('Point', layout=widgets.Layout(width='60px')),
            widgets.Label('X', layout=widgets.Layout(width='100px')),
            widgets.Label('Y', layout=widgets.Layout(width='130px'))
        ])
        
        default_x = self.default_x if self.default_x is not None else \
            np.array([30.75, 30.88, 31.00, 31.12, 31.25, 31.38, 31.50, 31.62, 31.75, 31.88])
        default_y = self.default_y if self.default_y is not None else \
            np.array([1056.6621, 1062.4049, 1059.5334, 1061.4478, 1060.0, 1061.0, 1060.5, 1061.5, 1060.8, 1061.2])
        
        rows = []
        for i in range(self.num_points):
            x_val = default_x[i] if i < len(default_x) else 30.0 + i * 0.12
            y_val = default_y[i] if i < len(default_y) else 1000.0
            
            x_input = widgets.FloatText(value=float(x_val), step=0.01,
                                        layout=widgets.Layout(width='100px'))
            y_input = widgets.FloatText(value=float(y_val), step=0.001,
                                        layout=widgets.Layout(width='130px'))
            
            self.x_inputs.append(x_input)
            self.y_inputs.append(y_input)
            
            rows.append(widgets.HBox([
                widgets.Label(str(i+1), layout=widgets.Layout(width='60px')),
                x_input,
                y_input
            ]))
        
        self.container.children = [header] + rows
    
    def update(self, n_points: int):
        self.num_points = n_points
        self._build_inputs()
    
    def get_data(self) -> Tuple[FloatArray, FloatArray]:
        x = np.array([widget.value for widget in self.x_inputs])
        y = np.array([widget.value for widget in self.y_inputs])
        return x, y

class ExtrapolationWidget:
    """Enhanced extrapolation widget with plotting."""
    
    def __init__(self, interpolator: ShiftedVandermondeInterpolator):
        self.interpolator = interpolator
        self.history = []
        self._plot_output = widgets.Output()
        self._build_widget()
    
    def _build_widget(self):
        """Build the extrapolation widget."""
        
        # Input section
        input_section = widgets.HBox([
            widgets.FloatText(
                description='X =',
                step=0.01,
                layout=widgets.Layout(width='200px')
            ),
            widgets.Button(
                description='Predict',
                button_style='primary',
                layout=widgets.Layout(width='120px')
            ),
            widgets.Button(
                description='Plot',
                button_style='success',
                layout=widgets.Layout(width='100px')
            )
        ])
        
        input_section.children[1].on_click(self._predict)
        input_section.children[2].on_click(self._plot)
        
        # Output sections
        self.result_output = widgets.Output()
        self.plot_output = widgets.Output()
        
        # Tab layout
        tabs = widgets.Tab([
            self.result_output,
            self.plot_output
        ])
        tabs.set_title(0, 'Results')
        tabs.set_title(1, 'Plots')
        
        self.container = widgets.VBox([
            widgets.HTML('<h4 style="margin: 10px 0 5px 0; color: #0d47a1;">Extrapolation</h4>'),
            widgets.HTML('<div style="font-size: 12px; color: #666;">Predict values outside the data range</div>'),
            input_section,
            tabs
        ])
    
    def _predict(self, btn):
        """Handle prediction button click."""
        with self.result_output:
            clear_output(wait=True)
            
            try:
                x_input = self.container.children[2].children[0].value
                result = self.interpolator.extrapolate(x_input)
                display(self._format_result(result))
                self.history.append((x_input, result.y_predicted))
                if len(self.history) > 10:
                    self.history = self.history[-10:]
                display(self._format_history())
            except Exception as e:
                display(widgets.HTML(f"""
                <div style="padding: 10px; background: #ffcdd2; border-radius: 4px; color: #b71c1c;">
                    Error: {str(e)}
                </div>
                """))
    
    def _plot(self, btn):
        """Generate and display plot."""
        with self.plot_output:
            clear_output(wait=True)
            
            try:
                x_input = self.container.children[2].children[0].value
                
                # Generate plot
                fig = self.interpolator.plot_extrapolation(x_input)
                
                # Display in widget
                plt.close(fig)
                display(fig)
                
            except Exception as e:
                display(widgets.HTML(f"""
                <div style="padding: 10px; background: #ffcdd2; border-radius: 4px; color: #b71c1c;">
                    Plot generation failed: {str(e)}
                </div>
                """))
    
    def _format_result(self, result: ExtrapolationResult) -> widgets.HTML:
        """Format extrapolation result."""
        colors = {
            ReliabilityLevel.HIGH: '#2e7d32',
            ReliabilityLevel.MEDIUM: '#f57c00',
            ReliabilityLevel.LOW: '#d32f2f',
            ReliabilityLevel.VERY_LOW: '#b71c1c',
            ReliabilityLevel.UNRELIABLE: '#880e4f'
        }
        color = colors.get(result.reliability_level, '#1a1a1a')
        
        html = f"""
        <div style="background: #f8f9fa; padding: 15px; border-radius: 8px; margin: 10px 0; border-left: 4px solid {color};">
            <div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 10px;">
                <div>
                    <div style="font-size: 11px; color: #666;">X Target</div>
                    <div style="font-size: 16px; font-weight: bold;">{result.x_target:.6f}</div>
                </div>
                <div>
                    <div style="font-size: 11px; color: #666;">Prediction</div>
                    <div style="font-size: 20px; font-weight: bold; color: {color};">{result.y_predicted:.10f}</div>
                </div>
                <div>
                    <div style="font-size: 11px; color: #666;">Reliability</div>
                    <div style="font-size: 16px; font-weight: bold; color: {color};">{result.reliability_score*100:.1f}%</div>
                    <div style="font-size: 12px; color: #666;">{result.reliability_level.value}</div>
                </div>
            </div>
            
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px; margin-top: 10px;">
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">Method Used</div>
                    <div style="font-size: 14px; font-weight: bold; color: #0d47a1;">{result.method_used}</div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">Extrapolation Distance</div>
                    <div style="font-size: 14px; font-weight: bold; color: #0d47a1;">{result.extrapolation_distance:.2f}x</div>
                </div>
            </div>
            
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px; margin-top: 10px;">
                <div style="background: #e8f0fe; padding: 8px; border-radius: 4px;">
                    <div style="font-size: 11px; color: #666;">95% Confidence Interval</div>
                    <div style="font-size: 13px; font-weight: bold; color: #0d47a1;">
                        [{result.confidence_interval[0]:.6f}, {result.confidence_interval[1]:.6f}]
                    </div>
                </div>
                <div style="background: #e8f0fe; padding: 8px; border-radius: 4px;">
                    <div style="font-size: 11px; color: #666;">Method Predictions</div>
                    <div style="font-size: 12px; font-family: monospace;">
                        {', '.join([f"{k}: {v:.6f}" for k, v in result.method_predictions.items() if v is not None])}
                    </div>
                </div>
            </div>
            
            <div style="margin-top: 8px; font-size: 11px; color: #999;">
                Time: {result.execution_time*1000:.2f}ms
            </div>
            
            {f'<div style="margin-top: 10px; padding: 10px; background: #ffcdd2; border-radius: 4px; border-left: 3px solid #b71c1c; color: #b71c1c;">{result.warning}</div>' if result.warning else ''}
            
            <div style="margin-top: 10px;">
                <button onclick="this.nextElementSibling.click()" style="padding: 6px 12px; background: #0d47a1; color: white; border: none; border-radius: 4px; cursor: pointer;">
                    📊 Generate Plot
                </button>
            </div>
        </div>
        """
        
        return widgets.HTML(html)
    
    def _format_history(self) -> widgets.HTML:
        """Format prediction history."""
        if not self.history:
            return widgets.HTML('')
        
        rows = "".join([
            f"<tr><td>{x:.6f}</td><td>{y:.10f}</td></tr>"
            for x, y in self.history
        ])
        
        return widgets.HTML(f"""
        <div style="margin-top: 10px;">
            <div style="font-size: 12px; color: #666;">Recent Predictions</div>
            <table style="width: auto; border-collapse: collapse; font-size: 12px;">
                <thead>
                    <tr style="background: #e8f0fe;">
                        <th style="padding: 4px 10px;">X</th>
                        <th style="padding: 4px 10px;">Y</th>
                    </tr>
                </thead>
                <tbody>
                    {rows}
                </tbody>
            </table>
        </div>
        """)

# ===================================================================
# RESULT DISPLAY
# ===================================================================

class ResultDisplay:
    """Display interpolation results with plotting."""
    
    @staticmethod
    def display_result(result: InterpolationResult, formatter: PolynomialFormatter,
                       interpolator: ShiftedVandermondeInterpolator = None):
        """Display complete results."""
        if result.status == SolverStatus.ERROR:
            display(widgets.HTML(f"""
            <div style="padding: 15px; background: #ffcdd2; border-radius: 4px; border-left: 5px solid #b71c1c;">
                <b>Error:</b> {result.error_message}
            </div>
            """))
            return
        
        # Create tabs
        tabs = widgets.Tab([])
        
        # Tab 1: Diagnostics
        diag_output = widgets.Output()
        with diag_output:
            display(ResultDisplay._diagnostics_widget(result))
        tabs.children = list(tabs.children) + [diag_output]
        tabs.set_title(len(tabs.children)-1, 'Diagnostics')
        
        # Tab 2: Polynomial
        poly_output = widgets.Output()
        with poly_output:
            display(ResultDisplay._polynomial_widget(result, formatter))
        tabs.children = list(tabs.children) + [poly_output]
        tabs.set_title(len(tabs.children)-1, 'Polynomial')
        
        # Tab 3: Verification
        verif_output = widgets.Output()
        with verif_output:
            display(ResultDisplay._verification_widget(result))
        tabs.children = list(tabs.children) + [verif_output]
        tabs.set_title(len(tabs.children)-1, 'Verification')
        
        # Tab 4: Evaluation
        eval_output = widgets.Output()
        with eval_output:
            display(ResultDisplay._evaluation_widget(result))
        tabs.children = list(tabs.children) + [eval_output]
        tabs.set_title(len(tabs.children)-1, 'Evaluation')
        
        # Tab 5: Plot (if interpolator available)
        if interpolator is not None and interpolator.has_result:
            plot_output = widgets.Output()
            with plot_output:
                try:
                    fig = interpolator.plot_interpolation()
                    plt.close(fig)
                    display(fig)
                except Exception as e:
                    display(widgets.HTML(f"<div style='color: #b71c1c;'>Plot generation failed: {str(e)}</div>"))
            tabs.children = list(tabs.children) + [plot_output]
            tabs.set_title(len(tabs.children)-1, 'Plot')
        
        display(tabs)
    
    @staticmethod
    def _diagnostics_widget(result: InterpolationResult) -> widgets.HTML:
        """Create diagnostics widget."""
        diag = result.diagnostics
        
        status_color = "#2e7d32" if not diag.rank_deficient and diag.condition_number < CONDITION_WARNING \
            else "#e65100" if diag.condition_number < CONDITION_CRITICAL else "#b71c1c"
        
        html = f"""
        <div style="background: #f8f9fa; padding: 15px; border-radius: 8px; margin: 10px 0; border-left: 4px solid #0d47a1;">
            <h4 style="margin: 0 0 10px 0; color: #0d47a1;">Numerical Diagnostics</h4>
            
            <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 8px; margin: 10px 0;">
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">Condition (log10)</div>
                    <div style="font-size: 18px; font-weight: bold; color: {status_color};">
                        {diag.condition_number_log10:.2f}
                    </div>
                    <div style="font-size: 11px; color: #999;">{diag.digits_lost:.1f} digits lost</div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">Residual Norm (L2)</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {diag.residual_norm_l2:.2e}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">Matrix Rank</div>
                    <div style="font-size: 16px; font-weight: bold; color: {'#2e7d32' if not diag.rank_deficient else '#b71c1c'};">
                        {diag.matrix_rank}/{diag.expected_rank}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">Backward Error</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {diag.backward_error:.2e}
                    </div>
                </div>
            </div>
            
            <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 8px; margin: 8px 0;">
                <div style="background: #ffffff; padding: 6px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 10px; color: #666;">Singular Ratio (log10)</div>
                    <div style="font-size: 14px; font-weight: bold; color: #0d47a1;">
                        {diag.singular_values.log10_ratio:.2f}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 6px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 10px; color: #666;">Effective Rank</div>
                    <div style="font-size: 14px; font-weight: bold; color: #0d47a1;">
                        {diag.singular_values.effective_rank}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 6px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 10px; color: #666;">Residual (L∞)</div>
                    <div style="font-size: 14px; font-weight: bold; color: #0d47a1;">
                        {diag.residual_norm_linf:.2e}
                    </div>
                </div>
            </div>
            
            <div style="margin-top: 8px; padding: 8px 10px; background: #e8f0fe; border-radius: 4px; font-size: 12px; color: #555; display: flex; gap: 20px; flex-wrap: wrap;">
                <span>Shift: {result.shift_value:.6f}</span>
                <span>Scale: {result.scale_value:.6f}</span>
                <span>Degree: {result.degree}</span>
                <span>Method: {result.method}</span>
                <span>Points: {len(result.x_data)}</span>
            </div>
        </div>
        """
        
        return widgets.HTML(html)
    
    @staticmethod
    def _polynomial_widget(result: InterpolationResult, 
                           formatter: PolynomialFormatter) -> widgets.HTML:
        """Create polynomial display widget."""
        shifted_str = formatter.format_shifted(
            result.shifted_coefficients, result.shift_value, result.scale_value
        )
        
        expanded = result.expanded_coefficients
        if expanded is None:
            expanded_str = "Not expanded (evaluating in shifted basis)"
        else:
            expanded_str = formatter.format_expanded(expanded)
        
        html = f"""
        <div style="background: #f8f9fa; padding: 15px; border-radius: 8px; margin: 10px 0; border-left: 4px solid #0d47a1;">
            <h4 style="margin: 0 0 10px 0; color: #0d47a1;">Polynomial Representation</h4>
            
            <div style="background: #e8f0fe; padding: 12px; border-radius: 4px; margin: 8px 0; border: 1px solid #90caf9;">
                <b>Shifted Form (Numerically Stable):</b><br>
                <span style="font-family: 'Courier New', monospace; font-size: 13px;">
                    {shifted_str}
                </span>
            </div>
            
            <div style="background: #ffffff; padding: 12px; border-radius: 4px; margin: 8px 0; border: 1px solid #e0e0e0;">
                <b>Expanded Form (Original Basis):</b><br>
                <span style="font-family: 'Courier New', monospace; font-size: 13px;">
                    {expanded_str}
                </span>
            </div>
        </div>
        """
        
        return widgets.HTML(html)
    
    @staticmethod
    def _verification_widget(result: InterpolationResult) -> widgets.HTML:
        """Create verification widget."""
        verif = result.verification
        
        html = f"""
        <div style="background: #f8f9fa; padding: 15px; border-radius: 8px; margin: 10px 0; border-left: 4px solid #0d47a1;">
            <h4 style="margin: 0 0 10px 0; color: #0d47a1;">Verification</h4>
            
            <div style="display: grid; grid-template-columns: repeat(5, 1fr); gap: 8px; margin: 8px 0;">
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">Max Error</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {verif.max_absolute_error:.2e}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">RMSE</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {verif.rmse:.2e}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">R²</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {verif.r_squared:.6f}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">L1 Norm</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {result.diagnostics.residual_norm_l1:.2e}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">L∞ Norm</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {result.diagnostics.residual_norm_linf:.2e}
                    </div>
                </div>
            </div>
        </div>
        """
        
        return widgets.HTML(html)
    
    @staticmethod
    def _evaluation_widget(result: InterpolationResult) -> widgets.VBox:
        """Create evaluation widget."""
        evaluator = ShiftedVandermondeInterpolator()
        evaluator._last_result = result
        
        history_output = widgets.Output()
        history = deque(maxlen=MAX_HISTORY)
        
        eval_input = widgets.FloatText(
            value=float(np.mean(result.x_data)),
            step=0.01,
            description='X =',
            layout=widgets.Layout(width='200px')
        )
        
        eval_result = widgets.HTML('<span style="font-size: 18px; font-weight: bold; color: #0d47a1;">= </span>')
        
        def evaluate(b):
            x_val = eval_input.value
            y_val = evaluator.evaluate(np.array([x_val]), use_shifted=True)[0]
            eval_result.value = f'<span style="font-size: 18px; font-weight: bold; color: #0d47a1;">= {y_val:.10f}</span>'
            
            history.append((x_val, y_val))
            
            with history_output:
                clear_output(wait=True)
                if history:
                    html = """
                    <div style="margin-top: 5px;">
                        <b>Evaluation History:</b>
                        <table style="width: auto; min-width: 200px; border-collapse: collapse; font-size: 13px;">
                            <thead>
                                <tr style="background: #0d47a1; color: #ffffff;">
                                    <th style="padding: 4px 10px; text-align: center;">X</th>
                                    <th style="padding: 4px 10px; text-align: center;">Y</th>
                                </tr>
                            </thead>
                            <tbody>
                    """
                    for x, y in history:
                        html += f"""
                            <tr style="border-bottom: 1px solid #e0e0e0;">
                                <td style="padding: 4px 10px; text-align: center;">{x:.6f}</td>
                                <td style="padding: 4px 10px; text-align: center;">{y:.10f}</td>
                            </tr>
                        """
                    html += "</tbody></table></div>"
                    display(widgets.HTML(html))
        
        eval_button = widgets.Button(
            description='Evaluate',
            button_style='primary',
            layout=widgets.Layout(width='120px')
        )
        eval_button.on_click(evaluate)
        
        return widgets.VBox([
            widgets.HTML('<h4 style="margin: 10px 0 5px 0; color: #0d47a1;">Polynomial Evaluation</h4>'),
            widgets.HBox([eval_input, eval_button, eval_result]),
            history_output
        ])

# ===================================================================
# MAIN APPLICATION
# ===================================================================

class InterpolationApplication:
    """Main application controller with interpolation and extrapolation."""
    
    def __init__(self):
        self.interpolator = ShiftedVandermondeInterpolator(
            shift_method=ShiftMethod.MEAN,
            scale_data=True,
            expansion_mode=ExpansionMode.BOTH
        )
        self.formatter = PolynomialFormatter()
        self.data_input = DataInputWidget()
        self.output = widgets.Output()
        self._extrapolation_widget = None
        self._build_ui()
    
    def _build_ui(self):
        """Build user interface."""
        display(HTML("""
        <style>
            .container {
                font-family: 'Times New Roman', serif;
                max-width: 1200px;
                margin: 0 auto;
                padding: 20px;
                background: #ffffff;
                color: #1a1a1a;
            }
            .header {
                background: #0d47a1;
                padding: 15px 20px;
                border-radius: 8px;
                text-align: center;
                margin-bottom: 20px;
            }
            .header h1 {
                color: #ffffff;
                font-size: 24px;
                margin: 0;
                font-weight: normal;
            }
            .header p {
                color: #e3f2fd;
                font-size: 13px;
                margin: 5px 0 0 0;
            }
            .panel {
                background: #f8f9fa;
                padding: 15px 20px;
                border-radius: 8px;
                margin: 10px 0;
                border-left: 4px solid #0d47a1;
            }
            .btn {
                padding: 8px 20px;
                border: none;
                border-radius: 4px;
                cursor: pointer;
                font-family: 'Times New Roman', serif;
                font-size: 14px;
            }
            .btn-primary {
                background: #0d47a1;
                color: #ffffff;
            }
            .btn-primary:hover {
                background: #1565c0;
            }
            .btn-success {
                background: #2e7d32;
                color: #ffffff;
            }
            .btn-success:hover {
                background: #388e3c;
            }
            .btn-secondary {
                background: #e0e0e0;
                color: #1a1a1a;
            }
            .btn-secondary:hover {
                background: #bdbdbd;
            }
            .widget-label {
                color: #1a1a1a !important;
            }
            .tab-content {
                padding: 10px 0;
            }
        </style>
        <div class="container">
            <div class="header">
                <h1>Shifted Vandermonde Interpolation & Extrapolation</h1>
                <p>Scientific Computing Implementation with Visualization</p>
            </div>
        """))
        
        # Input panel
        display(HTML('<div class="panel">'))
        display(HTML('<h3 style="margin: 0 0 10px 0; color: #0d47a1;">Data Input</h3>'))
        
        self.num_points = widgets.IntSlider(
            value=4, min=2, max=10, step=1,
            description='Points:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='300px')
        )
        
        self.update_btn = widgets.Button(
            description='Update Table',
            button_style='primary',
            layout=widgets.Layout(width='120px')
        )
        self.update_btn.add_class('btn btn-secondary')
        self.update_btn.on_click(self._update_points)
        
        display(widgets.HBox([self.num_points, self.update_btn]))
        
        self.data_container = widgets.VBox([self.data_input.container])
        display(self.data_container)
        display(HTML('</div>'))
        
        # Controls panel
        display(HTML('<div class="panel">'))
        display(HTML('<h3 style="margin: 0 0 10px 0; color: #0d47a1;">Solver Configuration</h3>'))
        
        self.shift_method = widgets.RadioButtons(
            options=['mean', 'first', 'none'],
            value='mean',
            description='Shift method:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='250px')
        )
        display(self.shift_method)
        
        self.use_lstsq = widgets.Checkbox(
            value=False,
            description='Least squares fitting',
            style={'description_width': 'initial'}
        )
        display(self.use_lstsq)
        
        buttons = widgets.HBox([
            widgets.Button(
                description='Interpolate',
                button_style='success',
                layout=widgets.Layout(width='150px', height='36px')
            ),
            widgets.Button(
                description='Clear All',
                button_style='primary',
                layout=widgets.Layout(width='100px', height='36px')
            ),
            widgets.Button(
                description='📊 Plot Interpolation',
                button_style='warning',
                layout=widgets.Layout(width='170px', height='36px')
            )
        ])
        buttons.children[0].add_class('btn btn-success')
        buttons.children[0].on_click(self._interpolate)
        buttons.children[1].add_class('btn btn-secondary')
        buttons.children[1].on_click(self._clear)
        buttons.children[2].add_class('btn btn-primary')
        buttons.children[2].on_click(self._plot_interpolation)
        display(buttons)
        
        display(HTML('</div>'))
        
        # Output
        display(HTML("<hr style='border: 2px solid #0d47a1;'>"))
        display(self.output)
        display(HTML('</div>'))
    
    def _update_points(self, btn):
        self.data_input.update(self.num_points.value)
        self.data_container.children = [self.data_input.container]
    
    def _plot_interpolation(self, btn):
        """Generate interpolation plot."""
        if not self.interpolator.has_result:
            with self.output:
                clear_output(wait=True)
                display(widgets.HTML("""
                <div style="padding: 15px; background: #fff3e0; border-radius: 4px; border-left: 4px solid #e65100;">
                    Please interpolate first before plotting.
                </div>
                """))
            return
        
        with self.output:
            # Create a new output for the plot
            plot_out = widgets.Output()
            display(plot_out)
            with plot_out:
                try:
                    fig = self.interpolator.plot_interpolation()
                    plt.close(fig)
                    display(fig)
                except Exception as e:
                    display(widgets.HTML(f"""
                    <div style="padding: 15px; background: #ffcdd2; border-radius: 4px; border-left: 4px solid #b71c1c;">
                        Plot generation failed: {str(e)}
                    </div>
                    """))
    
    def _interpolate(self, btn):
        with self.output:
            clear_output(wait=True)
            
            x_data, y_data = self.data_input.get_data()
            
            method_name = self.shift_method.value
            if method_name == 'mean':
                self.interpolator.shift_method = ShiftMethod.MEAN
            elif method_name == 'first':
                self.interpolator.shift_method = ShiftMethod.FIRST
            else:
                self.interpolator.shift_method = ShiftMethod.NONE
            
            display(widgets.HTML('<div style="padding: 10px; color: #0d47a1;">Computing interpolation...</div>'))
            
            if self.use_lstsq.value:
                result = self.interpolator.fit(x_data, y_data)
            else:
                result = self.interpolator.interpolate(x_data, y_data)
            
            clear_output(wait=True)
            
            if result.status == SolverStatus.ERROR:
                display(widgets.HTML(f"""
                <div style="padding: 15px; background: #ffcdd2; border-radius: 4px; border-left: 5px solid #b71c1c;">
                    <b>Error:</b> {result.error_message}
                </div>
                """))
                return
            
            # Display results with tabs
            ResultDisplay.display_result(result, self.formatter, self.interpolator)
            
            # Add extrapolation widget
            display(widgets.HTML("<hr style='border: 1px solid #0d47a1; margin: 20px 0;'>"))
            self._extrapolation_widget = ExtrapolationWidget(self.interpolator)
            display(self._extrapolation_widget.container)
    
    def _clear(self, btn):
        with self.output:
            clear_output(wait=True)
            self._extrapolation_widget = None
            display(widgets.HTML('''
            <div style="padding: 20px; text-align: center; color: #666; background: #f8f9fa; border-radius: 8px;">
                Results cleared. Enter data and click "Interpolate" to begin.
            </div>
            '''))

# ===================================================================
# ENTRY POINT
# ===================================================================

if __name__ == "__main__":
    print("\n" + "="*70)
    print("SHIFTED VANDERMONDE POLYNOMIAL INTERPOLATION & EXTRAPOLATION")
    print("Scientific Computing Implementation with Visualization")
    print("="*70)
    print("\nFeatures:")
    print("  - Shifted and scaled Vandermonde basis")
    print("  - Exact interpolation and least-squares fitting")
    print("  - Adaptive extrapolation with polynomial capping")
    print("  - Comprehensive numerical diagnostics")
    print("  - Publication-quality plots with Matplotlib")
    print("  - Interactive extrapolation with confidence bands")
    print("  - Method comparison visualization")
    print("  - Reliability assessment gauge")
    print("  - Residual analysis plots")
    print("  - Tabbed interface for organized results")
    print("\nInitialization completed.")
    print("="*70 + "\n")
    
    app = InterpolationApplication()


SHIFTED VANDERMONDE POLYNOMIAL INTERPOLATION & EXTRAPOLATION
Scientific Computing Implementation with Visualization

Features:
  - Shifted and scaled Vandermonde basis
  - Exact interpolation and least-squares fitting
  - Adaptive extrapolation with polynomial capping
  - Comprehensive numerical diagnostics
  - Publication-quality plots with Matplotlib
  - Interactive extrapolation with confidence bands
  - Method comparison visualization
  - Reliability assessment gauge
  - Residual analysis plots
  - Tabbed interface for organized results

Initialization completed.



RadioButtons(description='Shift method:', layout=Layout(width='250px'), options=('mean', 'first', 'none'), sty…

Checkbox(value=False, description='Least squares fitting', style=CheckboxStyle(description_width='initial'))

Output()